# UCS420: Cognitive Computing
## Assignment 6 — NumPy-II

**Name:** Sawaransh Mittal  
**Batch:** 3P13  
**Roll No:** 1024160030

In [ ]:
import numpy as np
print("NumPy version:", np.__version__)

NumPy version: 2.4.4


---
## Q1. Sensor readings

In [ ]:
temperature = np.array([25, 28, 31, 35, 38, 27, 33, 40])
print("Original readings (°C):", temperature)

Original readings (°C): [25 28 31 35 38 27 33 40]


### (a) Add 2°C to every reading (vectorization)
The scalar `2` is broadcast across the whole array, so no loop is needed.

In [ ]:
corrected = temperature + 2
print("Corrected readings (°C):", corrected)

Corrected readings (°C): [27 30 33 37 40 29 35 42]


### (b) Convert Celsius to Fahrenheit:  F = (9/5)·C + 32

In [ ]:
fahrenheit = (9 / 5) * temperature + 32
print("Readings in °F:", fahrenheit)

Readings in °F: [ 77.   82.4  87.8  95.  100.4  80.6  91.4 104. ]


> **Note:** the conversion is applied to the original readings. To convert the *error-corrected* readings instead, use `(9/5) * corrected + 32`.

In [ ]:
fahrenheit_corrected = (9 / 5) * corrected + 32
print("Corrected readings in °F:", fahrenheit_corrected)

Corrected readings in °F: [ 80.6  86.   91.4  98.6 104.   84.2  95.  107.6]


### (c) Readings greater than 32°C (Boolean indexing)

In [ ]:
mask = temperature > 32
print("Boolean mask      :", mask)
print("Readings > 32 °C  :", temperature[mask])

Boolean mask      : [False False False  True  True False  True  True]
Readings > 32 °C  : [35 38 33 40]


### (d) Count of readings exceeding 32°C

In [ ]:
count = np.sum(temperature > 32)      # True counts as 1
print("Number of readings > 32 °C:", count)
print("Same result with count_nonzero:", np.count_nonzero(temperature > 32))

Number of readings > 32 °C: 4
Same result with count_nonzero: 4


### (e) Why vectorization and Boolean indexing are more efficient than loops

* **Vectorization** applies an operation to the *entire array at once*. The loop runs inside pre-compiled **C code**, not in the Python interpreter.
* A Python `for` loop pays overhead on every iteration: **dynamic type checking** of each element, **high-level object abstractions** (every list element is a full Python object) and **interpreted execution** line by line. NumPy arrays store elements of one fixed dtype in contiguous memory, so none of this per-element overhead is needed.
* **Boolean indexing** (`temperature[temperature > 32]`) builds a mask and filters in a single optimised step. It replaces the loop + `if` + `append` pattern.
* The code is also **shorter and more readable**, and has **fewer bugs** (no index handling or manual accumulation).
* The advantage grows with data size. With millions of readings, the loop version becomes very slow, while the vectorised version stays near C speed.

Quick demonstration below:

In [ ]:
import time
big = np.random.randint(20, 45, 2_000_000)

# loop version
t0 = time.time()
loop_count = 0
for v in big:
    if v > 32:
        loop_count += 1
t_loop = time.time() - t0

# vectorised version
t0 = time.time()
vec_count = np.sum(big > 32)
t_vec = time.time() - t0

print(f"Loop count      : {loop_count}  (time = {t_loop:.4f} s)")
print(f"Vectorised count: {vec_count}  (time = {t_vec:.6f} s)")
print(f"Vectorised version is ~{t_loop / t_vec:.0f}x faster")

Loop count      : 960009  (time = 0.2050 s)
Vectorised count: 960009  (time = 0.003696 s)
Vectorised version is ~55x faster


---
## Q2. Daily steps of four users over three days

In [ ]:
steps = np.array([
    [5000, 6200, 7100],
    [8000, 7500, 9000],
    [4500, 5100, 4800],
    [9000, 8500, 9500]])
print(steps)
print("Shape:", steps.shape)

[[5000 6200 7100]
 [8000 7500 9000]
 [4500 5100 4800]
 [9000 8500 9500]]
Shape: (4, 3)


### (a) Total steps recorded

In [ ]:
print("Total steps:", steps.sum())

Total steps: 84200


### (b) Mean number of steps

In [ ]:
print("Mean steps:", steps.mean())

Mean steps: 7016.666666666667


### (c) Maximum and minimum values

In [ ]:
print("Maximum:", steps.max())
print("Minimum:", steps.min())

Maximum: 9500
Minimum: 4500


### (d) Total steps for each day (`axis=0` → collapses rows, one result per column)

In [ ]:
day_totals = steps.sum(axis=0)
print("Day 1, Day 2, Day 3 totals:", day_totals)

Day 1, Day 2, Day 3 totals: [26500 27300 30400]


### (e) Total steps for each user (`axis=1` → collapses columns, one result per row)

In [ ]:
user_totals = steps.sum(axis=1)
print("User 1..4 totals:", user_totals)

User 1..4 totals: [18300 24500 14400 27000]


### (f) Position of the maximum value using `argmax()`
`argmax()` returns an index into the *flattened* array, so `np.unravel_index` converts it to a (row, column) position.

In [ ]:
flat_idx = steps.argmax()
row, col = np.unravel_index(flat_idx, steps.shape)
print("Flattened index of max:", flat_idx)
print(f"Max value {steps[row, col]} is at User {row + 1}, Day {col + 1}  (row={row}, col={col})")

Flattened index of max: 11
Max value 9500 is at User 4, Day 3  (row=3, col=2)


---
## Q3. Slicing, views, copies, reshaping and flattening

### (a) Create the array

In [ ]:
original = np.array([1, 2, 3, 4, 5, 6])
print("original:", original)

original: [1 2 3 4 5 6]


### (b) Slice from index 1 to 4
Using `original[1:4]` (stop index is excluded), exactly as in the lecture example `b = a[1:4]`.

In [ ]:
subset = original[1:4]
print("subset:", subset)

subset: [2 3 4]


### (c) Modify the first element of `subset` to 999
Slicing returns a **view**, so the change is reflected in `original` too.

In [ ]:
subset[0] = 999
print("original:", original)
print("subset  :", subset)
print("subset shares memory with original:", np.shares_memory(original, subset))

original: [  1 999   3   4   5   6]
subset  : [999   3   4]
subset shares memory with original: True


### (d) Slice using `.copy()`
A copy is an independent array, so modifying it does **not** affect `original`.

In [ ]:
copied = original[1:4].copy()
copied[0] = 500
print("original:", original)       # unchanged by this step (999 is from part c)
print("copied  :", copied)
print("copied shares memory with original:", np.shares_memory(original, copied))

original: [  1 999   3   4   5   6]
copied  : [500   3   4]
copied shares memory with original: False


### (e) Numbers 1 to 12 with `np.arange()` reshaped into 3 × 4

In [ ]:
matrix = np.arange(1, 13).reshape(3, 4)
print(matrix)

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]


### (f) Indexing and slicing the 3 × 4 matrix
(Rows 1–2 and columns 2–3 are taken as the 1st–2nd row and 2nd–3rd column; NumPy indices are 0-based, so this is `[0:2, 1:3]`.)

In [ ]:
print("First row   :", matrix[0])
print("Last row    :", matrix[-1])
print("Second column:", matrix[:, 1])
print("Rows 1-2, columns 2-3:\n", matrix[0:2, 1:3])

First row   : [1 2 3 4]
Last row    : [ 9 10 11 12]
Second column: [ 2  6 10]
Rows 1-2, columns 2-3:
 [[2 3]
 [6 7]]


### (g) Flattened versions with `flatten()` and `ravel()`

In [ ]:
flat = matrix.flatten()
rav = matrix.ravel()
print("flatten():", flat)
print("ravel()  :", rav)

flatten(): [ 1  2  3  4  5  6  7  8  9 10 11 12]
ravel()  : [ 1  2  3  4  5  6  7  8  9 10 11 12]


### (h) Modify an element of the `ravel()` result

In [ ]:
rav[0] = 100
print("ravel array :", rav)
print("Original matrix:\n", matrix)
print("Original affected?", matrix[0, 0] == 100)

ravel array : [100   2   3   4   5   6   7   8   9  10  11  12]
Original matrix:
 [[100   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]
Original affected? True


`ravel()` returns a **view** (when possible), so the original matrix **is affected**.

### (i) Modify an element of the `flatten()` result

In [ ]:
flat[1] = 200
print("flatten array:", flat)
print("Original matrix:\n", matrix)
print("Original affected?", matrix[0, 1] == 200)

flatten array: [  1 200   3   4   5   6   7   8   9  10  11  12]
Original matrix:
 [[100   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]
Original affected? False


`flatten()` always returns a **copy**, so the original matrix is **not affected**.

### (j) Shape, ndim, size and dtype of the 3 × 4 matrix

In [ ]:
print("shape:", matrix.shape)
print("ndim :", matrix.ndim)
print("size :", matrix.size)
print("dtype:", matrix.dtype)

shape: (3, 4)
ndim : 2
size : 12
dtype: int64


---
## Q4. Predicting the assistance score (Ordinary Least Squares)

In [ ]:
X = np.array([
    [6, 70, 3],
    [5, 50, 6],
    [8, 80, 2],
    [4, 30, 8]
])
y = np.array([40, 65, 30, 85])
print("X =\n", X)
print("y =", y)

X =
 [[ 6 70  3]
 [ 5 50  6]
 [ 8 80  2]
 [ 4 30  8]]
y = [40 65 30 85]


### (a) Shape and dimensions of X

In [ ]:
print("Shape :", X.shape, "-> 4 users (rows) x 3 features (columns)")
print("ndim  :", X.ndim)

Shape : (4, 3) -> 4 users (rows) x 3 features (columns)
ndim  : 2


### (b) Transpose `X.T`

In [ ]:
print("X.T =\n", X.T)
print("Shape of X.T:", X.T.shape)

X.T =
 [[ 6  5  8  4]
 [70 50 80 30]
 [ 3  6  2  8]]
Shape of X.T: (3, 4)


**What the transpose represents:** transposing swaps rows and columns, so element `(i, j)` of `X` becomes element `(j, i)` of `X.T`. Originally each row is a *user* and each column a *feature*. In `X.T` each row is a *feature* (sleep hours, activity level, stress level) and each column is a *user*. The shape changes from (4, 3) to (3, 4), which is what makes the product `X.T @ X` possible.

### (c) Matrix product `X.T @ X`

In [ ]:
XtX = X.T @ X
print(XtX)
print("Shape:", XtX.shape)

[[  141  1430    96]
 [ 1430 14700   910]
 [   96   910   113]]
Shape: (3, 3)


`X.T` is (3×4) and `X` is (4×3). The number of columns of the first equals the number of rows of the second, so multiplication is valid and the result is a (3×3) square matrix. This is the matrix we need to invert.

### (d) Matrix inverse with `np.linalg.inv()`
An inverse exists only for a square matrix with non-zero determinant, so we check that first.

In [ ]:
print("Determinant of X.T @ X:", np.linalg.det(XtX))
XtX_inv = np.linalg.inv(XtX)
print("Inverse:\n", XtX_inv)
print("\nCheck (XtX @ inverse ≈ identity):\n", np.round(XtX @ XtX_inv, 6))

Determinant of X.T @ X: 753700.000000011
Inverse:
 [[ 1.10521428 -0.09848746 -0.14581398]
 [-0.09848746  0.00891203  0.01190129]
 [-0.14581398  0.01190129  0.0368847 ]]

Check (XtX @ inverse ≈ identity):
 [[ 1.  0.  0.]
 [ 0.  1.  0.]
 [-0.  0.  1.]]


### (e) Ordinary Least Squares: β = (XᵀX)⁻¹ Xᵀ y

In [ ]:
beta = np.linalg.inv(X.T @ X) @ X.T @ y
print("Coefficients β:", beta)
print("Sleep hours coefficient   :", beta[0])
print("Activity level coefficient:", beta[1])
print("Stress level coefficient  :", beta[2])

Coefficients β: [-0.15921454  0.14083853 10.06302242]
Sleep hours coefficient   : -0.15921454159465398
Activity level coefficient: 0.14083852991906243
Stress level coefficient  : 10.063022422714587


In [ ]:
# Sanity check on the training data
y_hat = X @ beta
print("Actual   y:", y)
print("Predicted :", np.round(y_hat, 2))
print("Residuals :", np.round(y - y_hat, 2))

Actual   y: [40 65 30 85]
Predicted : [39.09 66.62 30.12 84.09]
Residuals : [ 0.91 -1.62 -0.12  0.91]


### (f) What the coefficients represent
Each coefficient in `β` is the **weight** of one feature in the linear model  
`score = β₁·sleep + β₂·activity + β₃·stress`.  
It tells us by how much the predicted assistance score changes for a **one-unit increase in that feature, keeping the other two fixed**.

* **β₁ (sleep hours) ≈ −0.16:** slightly negative, so more sleep is associated with a marginally *lower* assistance score. The effect is very small, so sleep has almost no influence in this data.
* **β₂ (activity level) ≈ +0.14:** slightly positive. Each extra unit of activity raises the predicted score by about 0.14. This looks small, but activity values are large (30–80), so over its full range it contributes roughly 4–11 points.
* **β₃ (stress level) ≈ +10.06:** strongly positive. Each extra unit of stress raises the predicted assistance score by about 10 points, so **stress is the dominant factor**.

The magnitude shows how strongly a feature influences the score, though coefficients cannot be compared directly unless the features are on a similar scale. Note also that this model has **no intercept (bias) term**, and it is fitted on only 4 samples, so it is only a simple illustration.

### (g) Prediction for a new user

In [ ]:
new_user = np.array([5, 40, 7])
prediction = new_user @ beta
print("Predicted assistance score for new user:", round(float(prediction), 2))

Predicted assistance score for new user: 75.28


The prediction is the dot product of the new user's features with the learned coefficients: `5·β₁ + 40·β₂ + 7·β₃`.

---
**End of Assignment 6**